# TrustRAG — Multi-Domain Indian Legal RAG (Ingestion Notebook)

Three domains, one shared pipeline, tagged by `domain` metadata so retrieval can be filtered per case-file:

| Domain | Dataset | Source |
|---|---|---|
| **Consumer** | GeneralQA / SectoralQA / Judgments (Grahak-Nyay) | `github.com/ShreyGanatra/GrahakNyay` |
| **Bail** | IBPS bail judgments (150k+, check release status) or HLDC (public now) | `arxiv.org/abs/2508.07592` / HLDC on Hugging Face |
| **Matrimonial** | IMLJD (3,613 SC + Karnataka HC judgments) | `huggingface.co/datasets/joyboseroy/imljd` |

Run Section 1 once, then Sections 2–4 (one per domain) independently — you don't need all three datasets ready to start testing the pipeline on whichever one downloads cleanly first.


## 1. Environment setup

In [ ]:
!pip install -q datasets huggingface_hub pandas sentence-transformers rank-bm25 chromadb openai groq gitpython 


In [ ]:
import os, re, json, uuid
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
CHROMA_DIR = DATA_DIR / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

import chromadb
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "law-ai/InLegalBERT"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name="legal_chunks_multidomain",
    metadata={"hnsw:space": "cosine"}
)

def index_records(records, domain: str, batch_size: int = 32):
    """records: list of dicts with keys 'text' and 'citation'. Tags every chunk with its domain."""
    for i in range(0, len(records), batch_size):
        batch = records[i:i+batch_size]
        embeddings = embedder.encode([r["text"] for r in batch]).tolist()
        collection.upsert(
            ids=[str(uuid.uuid4()) for _ in batch],
            documents=[r["text"] for r in batch],
            embeddings=embeddings,
            metadatas=[{"domain": domain, "citation": r["citation"]} for r in batch],
        )
    print(f"Indexed {len(records)} chunks for domain='{domain}'. Collection total: {collection.count()}")


## 2. Domain: Consumer Grievance Redressal (Grahak-Nyay)

Clone the repo and load the QA + judgment datasets. Inspect the repo's actual folder structure after
cloning — file names/formats may have changed since this notebook was written, so the loader below
is a starting point you'll likely need to adjust to match what's actually in the repo.

In [ ]:
!git clone https://github.com/ShreyGanatra/GrahakNyay.git data/raw_consumer 2>/dev/null || echo "already cloned or clone failed — check the repo URL manually"


In [ ]:
# Inspect what actually came down — adjust the loader below to match
consumer_dir = DATA_DIR / "raw_consumer"
for p in consumer_dir.rglob("*"):
    if p.is_file() and p.suffix in (".json", ".csv", ".jsonl"):
        print(p)


In [ ]:
def load_consumer_records():
    records = []
    # Adjust these paths once you've seen the actual repo structure from the cell above.
    # Example pattern for a QA-style JSON file with {"question":..., "answer":..., "citation":...}:
    for json_path in consumer_dir.rglob("*.json"):
        try:
            data = json.loads(json_path.read_text(encoding="utf-8", errors="ignore"))
        except Exception as e:
            print(f"Skipped {json_path}: {e}")
            continue
        if isinstance(data, list):
            for item in data:
                text = item.get("answer") or item.get("context") or item.get("text") or ""
                citation = item.get("citation") or item.get("source") or json_path.stem
                if text.strip():
                    records.append({"text": text.strip(), "citation": citation})
    return records

consumer_records = load_consumer_records()
print(f"Loaded {len(consumer_records)} consumer records")
if consumer_records:
    index_records(consumer_records, domain="consumer")


## 3. Domain: Bail Adjudication

IBPS's full 150k-judgment set may not be public yet at the time you read this (check the arXiv page
first: `arxiv.org/abs/2508.07592`). If it isn't released, use **HLDC** as a public fallback — search
"HLDC Hindi Legal Document Corpus" on Hugging Face Datasets.

In [ ]:
from datasets import load_dataset

# Try IBPS first; fall back to HLDC if it isn't available yet
bail_records = []
try:
    bail_ds = load_dataset("IBPS-dataset-org/ibps")  # placeholder repo id — replace with the real one once released
    for row in bail_ds["train"]:
        text = row.get("facts") or row.get("judgment_text") or ""
        citation = row.get("case_id") or row.get("statute") or "IBPS case"
        if text.strip():
            bail_records.append({"text": text.strip(), "citation": str(citation)})
except Exception as e:
    print(f"IBPS not available yet ({e}). Trying HLDC as fallback...")
    try:
        hldc_ds = load_dataset("exploration-lab/HLDC")  # verify exact repo id on huggingface.co before running
        for row in hldc_ds["train"]:
            text = row.get("case_facts") or row.get("text") or ""
            citation = row.get("case_no") or "HLDC case"
            if text.strip():
                bail_records.append({"text": text.strip(), "citation": str(citation)})
    except Exception as e2:
        print(f"HLDC also failed ({e2}). Search huggingface.co/datasets directly and update the repo id above.")

print(f"Loaded {len(bail_records)} bail records")
if bail_records:
    index_records(bail_records[:2000], domain="bail")  # cap for a capstone-sized corpus; remove cap if you want more


## 4. Domain: Matrimonial Litigation (IMLJD)

In [ ]:
from datasets import load_dataset

matrimonial_records = []
try:
    imljd_ds = load_dataset("joyboseroy/imljd")
    split_name = list(imljd_ds.keys())[0]
    for row in imljd_ds[split_name]:
        text = row.get("summary") or row.get("case_text") or row.get("description") or ""
        citation = row.get("case_title") or row.get("case_id") or "IMLJD case"
        if text.strip():
            matrimonial_records.append({"text": text.strip(), "citation": str(citation)})
except Exception as e:
    print(f"Could not load IMLJD ({e}). Check https://huggingface.co/datasets/joyboseroy/imljd for the current schema.")

print(f"Loaded {len(matrimonial_records)} matrimonial records")
if matrimonial_records:
    index_records(matrimonial_records, domain="matrimonial")


## 5. Sanity check — query across domains

In [ ]:
def semantic_search(query: str, domain: str = None, top_k: int = 5):
    q_emb = embedder.encode([query]).tolist()
    where = {"domain": domain} if domain else None
    results = collection.query(query_embeddings=q_emb, n_results=top_k, where=where)
    hits = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": doc, "citation": meta["citation"], "domain": meta["domain"], "score": 1 - dist})
    return hits

for domain in ["consumer", "bail", "matrimonial"]:
    print(f"--- {domain} ---")
    for h in semantic_search("What are my rights in this case?", domain=domain, top_k=2):
        print(h["citation"], "|", h["text"][:100].replace("\n", " "))
    print()


---
This notebook only covers ingestion + indexing across three domains. The retrieval (BM25 + hybrid
+ reranking), generation, and citation verification logic is unchanged from `TrustRAG_starter_notebook.ipynb`
— just add a `domain` parameter to `hybrid_retrieve()` and pass it through to the ChromaDB `where` filter
as shown in `semantic_search()` above. The FastAPI backend (separate file) wires this together for the web UI.
